## Setup, Imports, and Triton Fixes

In [ ]:
!pip install -q --no-index --find-links /kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages datasets trl --ignore-installed

In [ ]:
import datasets
import trl
print('Successfully imported datasets version:', datasets.__version__)
print('Successfully imported trl version:', trl.__version__)

In [ ]:
import multiprocessing

multiprocessing.cpu_count()

In [ ]:
import os
import sys
import stat
import shutil
import gc
import re
import math
import zipfile
import types
import polars as pl
import torch
import torch.nn.functional as F
import kagglehub
from collections import Counter
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

# --- Force all trl optional-dep checks to False so it never tries to import them ---
import trl.import_utils as _trl_iu
for _attr in dir(_trl_iu):
    if _attr.startswith('is_') and _attr.endswith('_available') and callable(getattr(_trl_iu, _attr)):
        setattr(_trl_iu, _attr, lambda: False)

from trl import SFTTrainer, SFTConfig, GRPOTrainer, GRPOConfig

# --- Kaggle / Triton Environment Fixes ---
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell'
dst = '/tmp/ptxas-blackwell'
if os.path.exists(src):
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    import triton.backends.nvidia as nv_backend
    src_bin = os.path.join(os.path.dirname(nv_backend.__file__), 'bin')
    dst_bin = '/tmp/triton_nvidia_bin'
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
    os.environ['TRITON_PTXAS_PATH'] = dst

# --- Triton ptxas version override (before any model inference/training) ---
import triton.backends.nvidia.compiler as nv_compiler
os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = '/tmp/ptxas-blackwell'
nv_compiler.get_ptxas_version = lambda arch: '12.0'

# --- Hyperparameters ---
SFT_SAMPLES_PER_TYPE = 100    # SFT warmup: 6 * 100 = 600 samples
GRPO_SAMPLES_PER_TYPE = 100   # GRPO: 6 * 100 = 600 prompts (different data!)
LORA_RANK = 32
SFT_MAX_SEQ_LEN = 1024        # SFT: simple format, ~100 tokens avg
SFT_LR = 2e-4
GRPO_MAX_COMPLETION = 768      # GRPO: max generation length per completion
GRPO_NUM_GENERATIONS = 4       # GRPO: completions per prompt for ranking
GRPO_LR = 5e-6                 # RL: much lower LR than SFT
GRAD_ACCUM = 4

OUTPUT_DIR = '/kaggle/working/adapter'
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Phase 1: Data Loading & Preparation

In [ ]:
# Download model
MODEL_PATH = kagglehub.model_download('metric/nemotron-3-nano-30b-a3b-bf16/transformers/default')

# Load data
train_df = pl.read_csv('/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv')
print(f'Total training samples: {len(train_df)}')

# --- Type classification ---
def classify_type(prompt_text):
    p = prompt_text.lower()
    if 'bit manipulation' in p or '8-bit binary' in p: return 'bit_ops'
    elif 'encrypt' in p or 'decrypt' in p: return 'cipher'
    elif 'gravitational' in p or 'falling distance' in p: return 'gravity'
    elif 'numeral system' in p: return 'numeral'
    elif 'transformation rules' in p: return 'symbol'
    elif 'unit conversion' in p or 'convert the following measurement' in p: return 'unit_conv'
    return 'unknown'

train_df = train_df.with_columns(
    pl.col('prompt').map_elements(classify_type, return_dtype=pl.Utf8).alias('qtype')
)

# --- Stratified sampling: two SEPARATE sets ---
def stratified_sample(df, n_per_type, seed):
    dfs = []
    for qtype in df['qtype'].unique().to_list():
        subset = df.filter(pl.col('qtype') == qtype)
        n = min(n_per_type, len(subset))
        dfs.append(subset.sample(n=n, seed=seed))
    return pl.concat(dfs)

sft_df = stratified_sample(train_df, SFT_SAMPLES_PER_TYPE, seed=42)
grpo_df = stratified_sample(train_df, GRPO_SAMPLES_PER_TYPE, seed=123)
print(f'SFT samples: {len(sft_df)}, GRPO prompts: {len(grpo_df)}')

# --- Tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

METRIC_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

# --- Utility functions (match competition metric logic) ---
def extract_boxed_answer(text):
    matches = re.findall(r'\\boxed\{([^}]*)\}', text)
    for m in reversed(matches):
        if m.strip():
            return m.strip()
    return None

def answers_match(pred, gold):
    if pred is None:
        return False
    try:
        return math.isclose(float(pred), float(gold), rel_tol=1e-2, abs_tol=1e-5)
    except (ValueError, TypeError):
        pass
    return pred.strip().lower() == gold.strip().lower()

# --- Build SFT dataset ---
def build_sft_text(example):
    user_msg = example['prompt'] + METRIC_SUFFIX
    assistant_msg = f'\\boxed{{{example["answer"]}}}'
    messages = [
        {'role': 'user', 'content': user_msg},
        {'role': 'assistant', 'content': assistant_msg},
    ]
    for kwargs in [{'enable_thinking': True}, {}]:
        try:
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False, **kwargs
            )
            return {'text': text}
        except Exception:
            continue
    return {'text': f'<|im_start|>user\n{user_msg}<|im_end|>\n<|im_start|>assistant\n{assistant_msg}<|im_end|>'}

sft_dataset = Dataset.from_pandas(sft_df.drop('qtype').to_pandas())
sft_dataset = sft_dataset.map(build_sft_text, remove_columns=sft_dataset.column_names)
print(f'\nSFT dataset: {len(sft_dataset)} samples')
print(f'SFT example (first 500 chars):\n{sft_dataset[0]["text"][:500]}')

# --- Build GRPO dataset (prompts + gold answers for reward) ---
grpo_data = []
for row in grpo_df.iter_rows(named=True):
    grpo_data.append({
        'prompt': [{'role': 'user', 'content': row['prompt'] + METRIC_SUFFIX}],
        'answer': str(row['answer']),
    })
grpo_dataset = Dataset.from_list(grpo_data)
print(f'\nGRPO dataset: {len(grpo_dataset)} prompts')
print(f'GRPO example: {grpo_dataset[0]["prompt"][0]["content"][:200]}...')


## Phase 2: Model Loading & SFT Warmup
Quick SFT on 600 samples to teach output format. Matches V2 baseline (~0.65).
SFT adapter is saved as fallback before GRPO starts.

In [ ]:
# --- Load Model ---
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map='auto',
    trust_remote_code=True,
    dtype=torch.bfloat16,
)
print(f'Model loaded. Vocab: {len(tokenizer)}')

# --- Patches (must be done after every model load) ---
for name, mod in sys.modules.items():
    if 'modeling_nemotron_h' in name:
        mod.is_fast_path_available = False
        print(f'Patched {name}: slow path')

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

# --- LoRA ---
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=16,
    target_modules='all-linear',
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# --- SFT Training ---
sft_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=1,
    learning_rate=SFT_LR,
    logging_steps=10,
    bf16=True,
    max_grad_norm=1.0,
    optim='adamw_torch',
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    save_strategy='no',
    report_to='none',
    dataset_text_field='text',
    max_length=SFT_MAX_SEQ_LEN,
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': True},
)

sft_trainer = SFTTrainer(
    model=model,
    train_dataset=sft_dataset,
    processing_class=tokenizer,
    args=sft_args,
)

print(f'SFT: {len(sft_dataset)} samples x 1 epoch = ~{len(sft_dataset)//GRAD_ACCUM} steps')
print('Starting SFT warmup...')
sft_trainer.train()

# --- Save SFT adapter as fallback ---
model.save_pretrained(OUTPUT_DIR)
print(f'\n=== SFT adapter saved to {OUTPUT_DIR} (fallback submission) ===')

# --- Cleanup SFT trainer (keep model for GRPO) ---
del sft_trainer
gc.collect()
torch.cuda.empty_cache()
print('SFT cleanup done. Model stays in GPU for GRPO.')


## Phase 3: GRPO Reinforcement Learning
GRPO generates multiple completions per prompt, rewards correct answers,
and reinforces better reasoning strategies. This is the key to beating SFT.

In [ ]:
# --- Monkeypatch tokenizer for enable_thinking ---
# GRPO calls apply_chat_template internally to format prompts.
# We need enable_thinking=True to match evaluation (vLLM uses it).
_orig_apply_chat_template = tokenizer.apply_chat_template

def _apply_with_thinking(*args, **kwargs):
    kwargs.setdefault('enable_thinking', True)
    try:
        return _orig_apply_chat_template(*args, **kwargs)
    except TypeError:
        kwargs.pop('enable_thinking', None)
        return _orig_apply_chat_template(*args, **kwargs)

tokenizer.apply_chat_template = _apply_with_thinking

# --- Reward function ---
def correctness_reward(completions, answer, **kwargs):
    rewards = []
    for comp, gold in zip(completions, answer):
        pred = extract_boxed_answer(comp)
        if pred is not None and answers_match(pred, str(gold)):
            rewards.append(1.0)    # correct answer
        elif pred is not None:
            rewards.append(0.1)    # wrong but has \\boxed{}
        else:
            rewards.append(-0.5)   # no \\boxed{} at all
    return rewards

# --- Fix: GRPOTrainer expects model.warnings_issued dict (trl 0.24 compat) ---
if not hasattr(model, 'warnings_issued'):
    model.warnings_issued = {}

# --- GRPO Config ---
grpo_config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    
    # GRPO-specific
    num_generations=GRPO_NUM_GENERATIONS,      # 4 completions per prompt
    max_completion_length=GRPO_MAX_COMPLETION,  # 768 tokens max per completion
    beta=0.0,                                   # no KL penalty (saves ~60GB ref model)
    temperature=0.7,                            # generation diversity for exploration
    
    # Training
    per_device_train_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=1,
    learning_rate=GRPO_LR,
    
    # General
    bf16=True,
    max_grad_norm=0.5,
    optim='adamw_torch',
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    logging_steps=5,
    save_strategy='no',
    report_to='none',
    
    # Memory
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': True},
)

# --- Create GRPO Trainer ---
grpo_trainer = GRPOTrainer(
    model=model,
    reward_funcs=[correctness_reward],
    args=grpo_config,
    train_dataset=grpo_dataset,
    processing_class=tokenizer,
)

print(f'GRPO Training Plan:')
print(f'  Prompts: {len(grpo_dataset)}')
print(f'  Generations per prompt: {GRPO_NUM_GENERATIONS}')
print(f'  Max completion tokens: {GRPO_MAX_COMPLETION}')
print(f'  Temperature: 0.7 (diverse exploration)')
print(f'  LR: {GRPO_LR} (10x lower than SFT for RL stability)')
print(f'  Beta: 0.0 (no KL penalty, saves memory)')
print(f'  Optimizer steps: ~{len(grpo_dataset)//GRAD_ACCUM}')
print('Starting GRPO training...')
grpo_trainer.train()
print('\n=== GRPO training complete! ===')

## Save and Package Submission

In [ ]:
# Save final GRPO adapter (overwrites SFT fallback)
grpo_trainer.model.save_pretrained(OUTPUT_DIR)
print(f'Final GRPO adapter saved to {OUTPUT_DIR}:')
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f'  {f} ({size/1024:.1f} KB)')

# Package into submission.zip
zip_path = '/kaggle/working/submission.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(OUTPUT_DIR):
        fpath = os.path.join(OUTPUT_DIR, fname)
        zf.write(fpath, fname)

print(f'\nCreated {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)')

with zipfile.ZipFile(zip_path, 'r') as zf:
    names = zf.namelist()
    print(f'Contents: {names}')
    assert 'adapter_config.json' in names, 'Missing adapter_config.json!'
    print('submission.zip ready!')
